# Trends Report - Chart Prototyping

Interactive notebook for developing and previewing the matplotlib charts used in the
Organization Trends Report PDF. Each section mirrors a chart from `plot_builder.py`.

**Charts:**
1. Category Overview (horizontal bar)
2. Z-Score Heatmap (locations x categories, sorted by criticality)
3. Individual Overview (pie + wellness timeline) - top 20 locations only
3b. Individual Trends Table with severity highlighting
4. Category Health Distribution (stacked bar)
5. Location Time Series (line subplots)
6. Additional Locations Summary Table

**Report Features:**
- Locations sorted by criticality (critical > concerning > avg |z-score|)
- Top 20 locations get full analysis (LLM + charts)
- Remaining locations included as summary table with category scores
- Concerning/critical trends highlighted in location tables

In [ ]:
import datetime
import json
from io import BytesIO
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.dates as mdates

## Constants & Data Loading

In [ ]:
# ---------------------------------------------------------------------------
# Constants (from services.py)
# ---------------------------------------------------------------------------
ZSCORE_CONCERNING = 1.5
ZSCORE_CRITICAL = 2.5
MAX_DETAILED_LOCATIONS = 20

TREND_CATEGORY_LABELS = {
    "category.sleep": "Sleep",
    "category.bathroom": "Bathroom",
    "category.activity": "Activity",
    "category.social": "Social",
    "category.stability": "Stability",
    "category.energy": "Energy",
    "category.ambient": "Ambient",
    "category.care": "Care",
    "category.health": "Health",
    "category.summary": "Summary",
    "category.other": "Other",
}

# ---------------------------------------------------------------------------
# Color palettes (from plot_builder.py + PDF highlighting)
# ---------------------------------------------------------------------------
CATEGORY_COLORS = {
    "category.sleep": "#5B5EA6",
    "category.bathroom": "#9B59B6",
    "category.activity": "#2ECC71",
    "category.social": "#F39C12",
    "category.stability": "#E74C3C",
    "category.energy": "#1ABC9C",
    "category.ambient": "#3498DB",
    "category.care": "#E67E22",
    "category.health": "#E91E63",
    "category.summary": "#607D8B",
    "category.other": "#95A5A6",
}

HEALTH_COLORS = {
    "healthy": "#2ECC71",
    "concerning": "#F39C12",
    "critical": "#E74C3C",
}

# Row highlight colors for severity (used in PDF trends tables)
CONCERNING_FILL = "#FFF3E0"  # Light orange
CRITICAL_FILL = "#FFE0E0"    # Light red

In [ ]:
# ---------------------------------------------------------------------------
# Load real test data from tests/data/
# ---------------------------------------------------------------------------
DATA_DIR = Path("tests/data")

with open(DATA_DIR / "trends_metadata.json") as f:
    metadata = json.load(f)

with open(DATA_DIR / "trends.json") as f:
    raw_content = json.load(f)

# Extract trends time-series per location (strip the outer "trends" wrapper)
trends_data = {}
for location_id, loc_content in raw_content.items():
    location_trends = loc_content.get("trends", {})
    if location_trends:
        trends_data[str(location_id)] = location_trends

print(f"Loaded {len(trends_data)} locations, {len(metadata)} trend types in metadata")
for loc_id, days in trends_data.items():
    first_day = list(days.values())[0] if days else {}
    print(f"  Location {loc_id}: {len(days)} days, {len(first_day)} trends/day")

In [ ]:
# ---------------------------------------------------------------------------
# Derived data structures (mirrors report_builder.py logic)
# ---------------------------------------------------------------------------
def _get_latest_trends(days):
    """Collect most recent data point for each trend across ALL days."""
    latest = {}
    for day_ms in sorted(days.keys(), reverse=True):
        for trend_id, trend_data in days[day_ms].items():
            if trend_id not in latest:
                latest[trend_id] = trend_data
    return latest


def sort_by_criticality(trends_data):
    """Sort trends_data by location criticality (most critical first)."""
    def _criticality_key(location_id):
        days = trends_data[location_id]
        latest = _get_latest_trends(days)
        critical = 0
        concerning = 0
        abs_zscores = []
        for trend_data in latest.values():
            z = trend_data.get("zscore")
            if z is not None:
                abs_z = abs(z)
                abs_zscores.append(abs_z)
                if abs_z >= ZSCORE_CRITICAL:
                    critical += 1
                elif abs_z >= ZSCORE_CONCERNING:
                    concerning += 1
        avg_abs_z = sum(abs_zscores) / len(abs_zscores) if abs_zscores else 0
        return (-critical, -concerning, -avg_abs_z)

    sorted_ids = sorted(trends_data.keys(), key=_criticality_key)
    return {lid: trends_data[lid] for lid in sorted_ids}


def build_category_stats(trends_data, metadata):
    """Aggregate category statistics from trends data."""
    category_stats = {}
    for location_id, days in trends_data.items():
        latest_trends = _get_latest_trends(days)
        for trend_id, trend_data in latest_trends.items():
            meta = metadata.get(trend_id, {})
            category = meta.get("category", "category.other")
            if category not in category_stats:
                category_stats[category] = {
                    "trend_count": 0,
                    "location_count": 0,
                    "locations": set(),
                }
            category_stats[category]["locations"].add(location_id)
    for cat, stats in category_stats.items():
        stats["location_count"] = len(stats["locations"])
        stats["trend_count"] = len([
            t for t, m in metadata.items() if m.get("category") == cat
        ])
        del stats["locations"]
    return category_stats


def _extract_wellness_history(days, sorted_days, metadata):
    """
    Extract wellness score timeline for one location.
    Strategy: use trend.wellness_score if present, else synthesize from avg z-scores.
    """
    # Strategy 1: dedicated wellness trend
    wellness_trend_id = None
    for trend_id, meta in metadata.items():
        if "wellness" in meta.get("title", "").lower():
            wellness_trend_id = trend_id
            break
    if wellness_trend_id is None:
        for trend_id, meta in metadata.items():
            if meta.get("category") == "category.summary":
                wellness_trend_id = trend_id
                break

    if wellness_trend_id is not None:
        history = []
        for day_ms in sorted_days:
            trend_data = days[day_ms].get(wellness_trend_id)
            if trend_data and trend_data.get("value") is not None:
                history.append({"day_ms": int(day_ms), "value": trend_data["value"]})
        if history:
            return history

    # Strategy 2: synthetic from average z-scores
    history = []
    for day_ms in sorted_days:
        zscores = []
        for trend_data in days[day_ms].values():
            z = trend_data.get("zscore")
            if z is not None:
                zscores.append(abs(z))
        if zscores:
            avg_abs_z = sum(zscores) / len(zscores)
            wellness = max(0, min(100, 100 - avg_abs_z * 20))
            history.append({"day_ms": int(day_ms), "value": round(wellness, 1)})
    return history


def build_section2(trends_data, metadata):
    """Build per-location data for Section 2 charts."""
    individuals = {}
    for location_id, days in trends_data.items():
        if not days:
            continue
        sorted_days = sorted(days.keys())
        latest_trends = _get_latest_trends(days)

        current_trends = {}
        for trend_id, trend_data in latest_trends.items():
            meta = metadata.get(trend_id, {})
            current_trends[trend_id] = {
                "value": trend_data.get("value"),
                "display": trend_data.get("display", ""),
                "avg": trend_data.get("avg"),
                "std": trend_data.get("std"),
                "zscore": trend_data.get("zscore"),
                "category": meta.get("category", "category.other"),
                "title": meta.get("title", trend_id),
                "units": meta.get("units", ""),
            }

        # Historical time series
        historical = {}
        for day_ms in sorted_days:
            for trend_id, trend_data in days[day_ms].items():
                if trend_id not in historical:
                    historical[trend_id] = []
                historical[trend_id].append({
                    "day_ms": int(day_ms),
                    "value": trend_data.get("value"),
                    "display": trend_data.get("display", ""),
                })

        # Health distribution
        health_distribution = {"healthy": 0, "concerning": 0, "critical": 0}
        for trend_info in current_trends.values():
            zscore = trend_info.get("zscore")
            if zscore is not None:
                abs_z = abs(zscore)
                if abs_z >= ZSCORE_CRITICAL:
                    health_distribution["critical"] += 1
                elif abs_z >= ZSCORE_CONCERNING:
                    health_distribution["concerning"] += 1
                else:
                    health_distribution["healthy"] += 1

        # Wellness history
        wellness_history = _extract_wellness_history(days, sorted_days, metadata)

        individuals[str(location_id)] = {
            "current_trends": current_trends,
            "historical": historical,
            "health_distribution": health_distribution,
            "wellness_history": wellness_history,
        }
    return individuals


def build_section2_summary(trends_data, metadata):
    """Build lightweight summary for locations beyond MAX_DETAILED_LOCATIONS."""
    summary = {}
    for location_id, days in trends_data.items():
        if not days:
            continue
        latest_trends = _get_latest_trends(days)
        category_zscores = {}
        critical_count = 0
        concerning_count = 0
        total_trends = 0
        for trend_id, trend_data in latest_trends.items():
            total_trends += 1
            meta = metadata.get(trend_id, {})
            category = meta.get("category", "category.other")
            zscore = trend_data.get("zscore")
            if zscore is not None:
                abs_z = abs(zscore)
                if abs_z >= ZSCORE_CRITICAL:
                    critical_count += 1
                elif abs_z >= ZSCORE_CONCERNING:
                    concerning_count += 1
                if category not in category_zscores:
                    category_zscores[category] = []
                category_zscores[category].append(abs_z)
        category_scores = {}
        for category, zscores in category_zscores.items():
            category_scores[category] = round(sum(zscores) / len(zscores), 2)
        summary[str(location_id)] = {
            "category_scores": category_scores,
            "total_trends": total_trends,
            "critical_count": critical_count,
            "concerning_count": concerning_count,
        }
    return summary


def build_category_health(trends_data, metadata):
    """Build category health classification from z-scores."""
    trend_zscores = {}
    for location_id, days in trends_data.items():
        latest_trends = _get_latest_trends(days)
        for trend_id, trend_data in latest_trends.items():
            zscore = trend_data.get("zscore")
            if zscore is not None:
                if trend_id not in trend_zscores:
                    trend_zscores[trend_id] = []
                trend_zscores[trend_id].append(zscore)

    category_health = {}
    for trend_id, zscores in trend_zscores.items():
        meta = metadata.get(trend_id, {})
        category = meta.get("category", "category.other")
        if category not in category_health:
            category_health[category] = {"healthy": 0, "concerning": 0, "critical": 0}
        for z in zscores:
            abs_z = abs(z)
            if abs_z >= ZSCORE_CRITICAL:
                category_health[category]["critical"] += 1
            elif abs_z >= ZSCORE_CONCERNING:
                category_health[category]["concerning"] += 1
            else:
                category_health[category]["healthy"] += 1
    return category_health


# Sort by criticality (most critical first)
trends_data = sort_by_criticality(trends_data)

# Split into detailed and summary
all_location_ids = list(trends_data.keys())
detailed_ids = all_location_ids[:MAX_DETAILED_LOCATIONS]
summary_ids = all_location_ids[MAX_DETAILED_LOCATIONS:]
detailed_data = {lid: trends_data[lid] for lid in detailed_ids}
summary_data = {lid: trends_data[lid] for lid in summary_ids}

# Build derived structures (Section 1 & 3 use ALL locations)
category_stats = build_category_stats(trends_data, metadata)
section2_data = build_section2(detailed_data, metadata)
section2_summary = build_section2_summary(summary_data, metadata)
category_health = build_category_health(trends_data, metadata)

print(f"Total locations: {len(trends_data)}")
print(f"Detailed (top {MAX_DETAILED_LOCATIONS}): {len(detailed_ids)}")
print(f"Summary: {len(summary_ids)}")
print(f"Categories: {sorted(category_stats.keys())}")
print(f"\nDetailed locations (sorted by criticality):")
for loc_id in detailed_ids:
    loc = section2_data[loc_id]
    h = loc["health_distribution"]
    print(f"  Location {loc_id}: {len(loc['current_trends'])} trends, "
          f"health=[H:{h['healthy']} C:{h['concerning']} X:{h['critical']}], "
          f"{len(loc['wellness_history'])} wellness points")
if summary_ids:
    print(f"\nSummary locations:")
    for loc_id in summary_ids:
        s = section2_summary[loc_id]
        print(f"  Location {loc_id}: {s['total_trends']} trends, "
              f"critical={s['critical_count']}, concerning={s['concerning_count']}")

---
## 1. Category Overview Chart

Horizontal bar chart showing trend type counts and location coverage by category.
Used in Section 1 of the PDF report.

In [ ]:
def generate_category_overview_chart(category_stats, metadata):
    """Horizontal bar chart: trend counts and location coverage by category."""
    if not category_stats:
        print("No category data available")
        return

    categories = []
    trend_counts = []
    location_counts = []
    colors = []

    for cat, stats in sorted(category_stats.items()):
        label = TREND_CATEGORY_LABELS.get(cat, cat)
        categories.append(label)
        trend_counts.append(stats.get("trend_count", 0))
        location_counts.append(stats.get("location_count", 0))
        colors.append(CATEGORY_COLORS.get(cat, "#95A5A6"))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, max(3, len(categories) * 0.5)))

    y_pos = range(len(categories))

    ax1.barh(y_pos, trend_counts, color=colors, height=0.6)
    ax1.set_yticks(y_pos)
    ax1.set_yticklabels(categories, fontsize=9)
    ax1.set_xlabel("Trend Types", fontsize=9)
    ax1.set_title("Trends by Category", fontsize=11, fontweight="bold")
    ax1.invert_yaxis()

    ax2.barh(y_pos, location_counts, color=colors, height=0.6)
    ax2.set_yticks(y_pos)
    ax2.set_yticklabels(categories, fontsize=9)
    ax2.set_xlabel("Locations", fontsize=9)
    ax2.set_title("Location Coverage", fontsize=11, fontweight="bold")
    ax2.invert_yaxis()

    plt.tight_layout()
    plt.show()


generate_category_overview_chart(category_stats, metadata)

---
## 2. Z-Score Heatmap

Heatmap of average absolute z-scores with rows=locations and columns=categories.
Used in Section 1 of the PDF report.

In [ ]:
def generate_zscore_heatmap(section2_data, metadata):
    """Heatmap: avg |z-score| by location and category."""
    if not section2_data:
        print("No location data available")
        return

    all_categories = set()
    for loc_id, loc_data in section2_data.items():
        for trend_id, trend_info in loc_data.get("current_trends", {}).items():
            all_categories.add(trend_info.get("category", "category.other"))

    categories = sorted(all_categories)
    location_ids = sorted(section2_data.keys())

    if not categories or not location_ids:
        print("Insufficient data for heatmap")
        return

    matrix = []
    for loc_id in location_ids:
        row = []
        current = section2_data[loc_id].get("current_trends", {})
        for cat in categories:
            zscores = [
                abs(t.get("zscore", 0))
                for t in current.values()
                if t.get("category") == cat and t.get("zscore") is not None
            ]
            avg_z = sum(zscores) / len(zscores) if zscores else 0
            row.append(avg_z)
        matrix.append(row)

    fig, ax = plt.subplots(figsize=(max(7, len(categories) * 1.0), max(3, len(location_ids) * 0.5)))

    cat_labels = [TREND_CATEGORY_LABELS.get(c, c) for c in categories]
    loc_labels = [f"Location {lid}" for lid in location_ids]

    im = ax.imshow(matrix, cmap="RdYlGn_r", aspect="auto", vmin=0, vmax=3)
    ax.set_xticks(range(len(cat_labels)))
    ax.set_xticklabels(cat_labels, fontsize=8, rotation=45, ha="right")
    ax.set_yticks(range(len(loc_labels)))
    ax.set_yticklabels(loc_labels, fontsize=8)
    ax.set_title("Z-Score Overview by Location & Category", fontsize=11, fontweight="bold")

    # Annotate cells with values
    for i in range(len(location_ids)):
        for j in range(len(categories)):
            val = matrix[i][j]
            color = "white" if val > 1.5 else "black"
            ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=8, color=color)

    cbar = fig.colorbar(im, ax=ax, shrink=0.8)
    cbar.set_label("Avg |Z-Score|", fontsize=9)

    plt.tight_layout()
    plt.show()


generate_zscore_heatmap(section2_data, metadata)

---
## 3. Individual Overview Charts

Side-by-side pie chart (health distribution) and wellness score timeline.
One pair per location in Section 2 of the PDF report (top 20 locations only).
Concerning/critical trends are highlighted in the trends table.

In [ ]:
def generate_individual_overview_charts(health_distribution, wellness_history):
    """Side-by-side pie chart + wellness score timeline for one location."""
    has_pie = any(v > 0 for v in health_distribution.values())
    has_timeline = bool(wellness_history)

    if not has_pie and not has_timeline:
        print("No individual overview data available")
        return

    fig, (ax_pie, ax_timeline) = plt.subplots(1, 2, figsize=(10, 3.5))

    # Left: Pie chart
    if has_pie:
        labels = []
        sizes = []
        colors = []
        for level in ("healthy", "concerning", "critical"):
            count = health_distribution.get(level, 0)
            if count > 0:
                labels.append(f"{level.capitalize()} ({count})")
                sizes.append(count)
                colors.append(HEALTH_COLORS[level])

        ax_pie.pie(
            sizes,
            labels=labels,
            colors=colors,
            autopct="%1.0f%%",
            startangle=90,
            textprops={"fontsize": 9},
        )
        ax_pie.set_title("Trend Health", fontsize=11, fontweight="bold")
    else:
        ax_pie.text(0.5, 0.5, "No health data", ha="center", va="center", fontsize=10, color="#999999")
        ax_pie.axis("off")

    # Right: Wellness score timeline
    if has_timeline:
        dates = []
        values = []
        for point in wellness_history:
            if point.get("value") is not None:
                try:
                    dt = datetime.datetime.fromtimestamp(int(point["day_ms"]) / 1000)
                    dates.append(dt)
                    values.append(point["value"])
                except (ValueError, TypeError, OSError):
                    continue

        if dates and values:
            ax_timeline.plot(dates, values, color="#2980B9", linewidth=2, marker="o", markersize=4)
            ax_timeline.fill_between(dates, values, alpha=0.1, color="#2980B9")
            ax_timeline.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))
            ax_timeline.xaxis.set_major_locator(mdates.AutoDateLocator())
            ax_timeline.tick_params(axis="x", labelsize=7, rotation=30)
            ax_timeline.tick_params(axis="y", labelsize=8)
            ax_timeline.set_ylabel("Score", fontsize=9)
            ax_timeline.grid(axis="y", alpha=0.3)
        ax_timeline.set_title("Wellness Score", fontsize=11, fontweight="bold")
    else:
        ax_timeline.text(0.5, 0.5, "No wellness data", ha="center", va="center", fontsize=10, color="#999999")
        ax_timeline.axis("off")

    plt.tight_layout()
    plt.show()


# Show charts for all locations
for loc_id in sorted(section2_data.keys()):
    loc = section2_data[loc_id]
    h = loc["health_distribution"]
    print(f"\n--- Location {loc_id} (H:{h['healthy']} C:{h['concerning']} X:{h['critical']}) ---")
    generate_individual_overview_charts(loc["health_distribution"], loc["wellness_history"])

---
## 3b. Individual Trends Table with Severity Highlighting

Matplotlib table showing per-trend data for a single location.
Rows are highlighted by z-score severity:
- **Light red** (`#FFE0E0`): critical (`|z| >= 2.5`)
- **Light orange** (`#FFF3E0`): concerning (`|z| >= 1.5`)
- No fill: healthy

Mirrors the fpdf2 trends table in Section 2 of the PDF report.

In [ ]:
import matplotlib.colors as mcolors


def generate_trends_table(current_trends, location_id=""):
    """
    Matplotlib table with severity-highlighted rows for a single location.
    Mirrors the fpdf2 trends table in Section 2 of the PDF report.

    Row colors:
    - Critical (|z| >= 2.5): light red
    - Concerning (|z| >= 1.5): light orange
    - Healthy: white
    """
    if not current_trends:
        print("No trend data available")
        return

    # Build table data: [Trend, Value, Avg, Std Dev, Z-Score]
    columns = ["Trend", "Value", "Avg", "Std Dev", "Z-Score"]
    cell_text = []
    row_colors = []

    for trend_id, info in current_trends.items():
        zscore = info.get("zscore")
        abs_z = abs(zscore) if zscore is not None else 0

        if abs_z >= ZSCORE_CRITICAL:
            row_colors.append(CRITICAL_FILL)
        elif abs_z >= ZSCORE_CONCERNING:
            row_colors.append(CONCERNING_FILL)
        else:
            row_colors.append("#FFFFFF")

        cell_text.append([
            info.get("title", trend_id),
            info.get("display", str(info.get("value", "--"))) or str(info.get("value", "--")),
            str(round(info["avg"], 2)) if info.get("avg") is not None else "--",
            str(round(info["std"], 2)) if info.get("std") is not None else "--",
            str(round(zscore, 2)) if zscore is not None else "--",
        ])

    fig_height = max(2, len(cell_text) * 0.35 + 1)
    fig, ax = plt.subplots(figsize=(10, fig_height))
    ax.axis("off")

    title = f"Location Analysis (#{location_id})" if location_id else "Location Trends"
    ax.set_title(title, fontsize=11, fontweight="bold", loc="left", pad=10)

    table = ax.table(
        cellText=cell_text,
        colLabels=columns,
        cellLoc="center",
        loc="center",
        colWidths=[0.30, 0.18, 0.18, 0.18, 0.16],
    )
    table.auto_set_font_size(False)
    table.set_fontsize(8)
    table.scale(1, 1.3)

    # Style header row
    for j in range(len(columns)):
        cell = table[0, j]
        cell.set_facecolor("#2980B9")
        cell.set_text_props(color="white", fontweight="bold")

    # Style data rows with severity colors
    for i, color in enumerate(row_colors):
        for j in range(len(columns)):
            cell = table[i + 1, j]
            cell.set_facecolor(color)
            # Bold the z-score column for non-healthy rows
            if color != "#FFFFFF" and j == 4:
                cell.set_text_props(fontweight="bold")

    plt.tight_layout()
    plt.show()


# Show highlighted trends table for the first (most critical) location
first_loc_id = list(section2_data.keys())[0]
loc = section2_data[first_loc_id]
h = loc["health_distribution"]
print(f"Location {first_loc_id} - H:{h['healthy']} C:{h['concerning']} X:{h['critical']}")
generate_trends_table(loc["current_trends"], location_id=first_loc_id)

---
## 4. Category Health Distribution

Stacked bar chart of healthy/concerning/critical counts per category.
Used in Section 3 of the PDF report.

In [ ]:
def generate_category_health_chart(category_health):
    """Stacked bar chart: healthy/concerning/critical by category."""
    if not category_health:
        print("No category health data available")
        return

    categories = sorted(category_health.keys())
    labels = [TREND_CATEGORY_LABELS.get(c, c) for c in categories]
    healthy = [category_health[c].get("healthy", 0) for c in categories]
    concerning = [category_health[c].get("concerning", 0) for c in categories]
    critical = [category_health[c].get("critical", 0) for c in categories]

    fig, ax = plt.subplots(figsize=(max(7, len(categories) * 1.0), 4.5))
    x = range(len(categories))
    width = 0.6

    ax.bar(x, healthy, width, label="Healthy", color=HEALTH_COLORS["healthy"])
    ax.bar(x, concerning, width, bottom=healthy, label="Concerning", color=HEALTH_COLORS["concerning"])
    bottom_critical = [h + c for h, c in zip(healthy, concerning)]
    ax.bar(x, critical, width, bottom=bottom_critical, label="Critical", color=HEALTH_COLORS["critical"])

    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=9, rotation=45, ha="right")
    ax.set_ylabel("Trend-Location Readings", fontsize=9)
    ax.set_title("Category Health Distribution", fontsize=11, fontweight="bold")
    ax.legend(fontsize=9)

    plt.tight_layout()
    plt.show()


generate_category_health_chart(category_health)

---
## 5. Location Time Series

Line chart subplots for one location's historical trend data.
Batched into pages of 5 subplots max. Used in Section 2 of the PDF report.

In [ ]:
MAX_PLOTS_PER_PAGE = 5


def generate_location_time_series(historical_trends, metadata, max_per_page=MAX_PLOTS_PER_PAGE):
    """Line chart subplots for one location's historical trends."""
    if not historical_trends:
        print("No historical data available")
        return

    sorted_trends = sorted(
        historical_trends.items(),
        key=lambda x: len(x[1]),
        reverse=True,
    )

    if not sorted_trends:
        print("No trend data available")
        return

    batches = [
        sorted_trends[i : i + max_per_page]
        for i in range(0, len(sorted_trends), max_per_page)
    ]

    for batch_idx, batch in enumerate(batches):
        n_plots = len(batch)
        fig, axes = plt.subplots(n_plots, 1, figsize=(9, 2.5 * n_plots), squeeze=False)

        for idx, (trend_id, data_points) in enumerate(batch):
            ax = axes[idx, 0]
            meta = metadata.get(trend_id, {})
            title = meta.get("title", trend_id)
            units = meta.get("units", "")
            category = meta.get("category", "category.other")
            color = CATEGORY_COLORS.get(category, "#3498DB")

            dates = []
            values = []
            for point in data_points:
                if point.get("value") is not None:
                    try:
                        dt = datetime.datetime.fromtimestamp(point["day_ms"] / 1000)
                        dates.append(dt)
                        values.append(point["value"])
                    except (ValueError, TypeError, OSError):
                        continue

            if dates and values:
                ax.plot(dates, values, color=color, linewidth=1.5, marker=".", markersize=4)
                ax.fill_between(dates, values, alpha=0.1, color=color)
                ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))
                ax.xaxis.set_major_locator(mdates.AutoDateLocator())
                ax.tick_params(axis="x", labelsize=7, rotation=30)
                ax.tick_params(axis="y", labelsize=8)

            label = f"{title}"
            if units:
                label += f" ({units})"
            ax.set_title(label, fontsize=10, fontweight="bold", loc="left")
            ax.grid(axis="y", alpha=0.3)

        plt.tight_layout()
        plt.show()


# Show time series for the location with the most trends (287960: 24 trends)
richest_loc_id = max(section2_data.keys(), key=lambda k: len(section2_data[k]["historical"]))
loc = section2_data[richest_loc_id]
print(f"--- Location {richest_loc_id}: {len(loc['historical'])} trends, 31 days ---")
generate_location_time_series(loc["historical"], metadata)

---
## 6. Additional Locations Summary Table

Compact table for locations beyond the top 20 (MAX_DETAILED_LOCATIONS).
Shows location ID, trend count, critical/concerning counts, and average |z-score|
per category. No LLM narratives or charts for these locations.

Mirrors the fpdf2 summary table in Section 2B of the PDF report.

In [ ]:
def generate_summary_table(section2_summary, detailed_count=0):
    """
    Matplotlib table for additional locations beyond MAX_DETAILED_LOCATIONS.
    Columns: Location, Trends, Critical, Concerning, then one col per category.

    Mirrors the fpdf2 summary table in Section 2B of the PDF report.
    """
    if not section2_summary:
        print("No summary locations (all locations fit within MAX_DETAILED_LOCATIONS)")
        return

    # Collect all categories across summary locations
    all_cats = set()
    for loc_data in section2_summary.values():
        all_cats.update(loc_data.get("category_scores", {}).keys())
    sorted_cats = sorted(all_cats)
    cat_labels = [TREND_CATEGORY_LABELS.get(c, c) for c in sorted_cats]

    # Build table
    columns = ["Location", "Trends", "Critical", "Concern."] + cat_labels
    cell_text = []

    for idx, (location_id, loc_data) in enumerate(section2_summary.items()):
        row = [
            f"#{detailed_count + idx + 1} - {location_id}",
            str(loc_data.get("total_trends", 0)),
            str(loc_data.get("critical_count", 0)),
            str(loc_data.get("concerning_count", 0)),
        ]
        scores = loc_data.get("category_scores", {})
        for cat in sorted_cats:
            val = scores.get(cat)
            row.append(str(round(val, 1)) if val is not None else "--")
        cell_text.append(row)

    fig_height = max(2, len(cell_text) * 0.35 + 1.5)
    fig_width = max(10, len(columns) * 1.2)
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    ax.axis("off")
    ax.set_title(
        f"Additional Locations Summary ({len(section2_summary)} locations)",
        fontsize=11, fontweight="bold", loc="left", pad=10,
    )

    # Column widths: wider for Location, narrower for data columns
    col_widths = [0.18] + [0.08] * 3 + [0.08] * len(sorted_cats)
    total = sum(col_widths)
    col_widths = [w / total for w in col_widths]

    table = ax.table(
        cellText=cell_text,
        colLabels=columns,
        cellLoc="center",
        loc="center",
        colWidths=col_widths,
    )
    table.auto_set_font_size(False)
    table.set_fontsize(7)
    table.scale(1, 1.3)

    # Style header row
    for j in range(len(columns)):
        cell = table[0, j]
        cell.set_facecolor("#2980B9")
        cell.set_text_props(color="white", fontweight="bold", fontsize=7)

    # Alternating row fills
    for i in range(len(cell_text)):
        bg = "#F5F5F5" if i % 2 == 0 else "#FFFFFF"
        for j in range(len(columns)):
            table[i + 1, j].set_facecolor(bg)

    plt.tight_layout()
    plt.show()


generate_summary_table(section2_summary, detailed_count=len(section2_data))